In [0]:
from pyspark.sql.functions import col, lit, when, expr, to_date, row_number, concat, trunc, last_day, date_format

from pyspark.sql.window import Window

In [0]:
base_path = "file:/Workspace/Users/rhuanjlvi@hotmail.com/"

products_path = f"{base_path}/products.csv"
sales_order_detail_path = f"{base_path}/sales-order-detail.csv"
sales_order_header_path = f"{base_path}/sales-order-header.csv"

raw_products_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "False")
    .csv(products_path)
)

raw_sales_order_detail_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "False")
    .csv(sales_order_detail_path)
)

raw_sales_order_header_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "False")
    .csv(sales_order_header_path)
)


In [0]:
display(raw_products_df.limit(10))

ProductID,ProductDesc,ProductNumber,MakeFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,Weight,WeightUnitMeasureCode,ProductCategoryName,ProductSubCategoryName
680,"HL Road Frame - Black, 58",FR-R92B-58,True,Black,500,375,1059.31,1431.5,58,CM,2.24,LB,null,Road Frames
706,"HL Road Frame - Red, 58",FR-R92R-58,True,Red,500,375,1059.31,1431.5,58,CM,2.24,LB,null,Road Frames
707,"Sport-100 Helmet, Red",HL-U509-R,False,Red,4,3,13.0863,34.99,null,null,null,null,null,Helmets
708,"Sport-100 Helmet, Black",HL-U509,False,Black,4,3,13.0863,34.99,null,null,null,null,null,Helmets
709,"Mountain Bike Socks, M",SO-B909-M,False,White,4,3,3.3963,9.5,M,null,null,null,null,Socks
710,"Mountain Bike Socks, L",SO-B909-L,False,White,4,3,3.3963,9.5,L,null,null,null,null,Socks
711,"Sport-100 Helmet, Blue",HL-U509-B,False,Blue,4,3,13.0863,34.99,null,null,null,null,null,Helmets
712,AWC Logo Cap,CA-1098,False,Multi,4,3,6.9223,8.99,null,null,null,null,null,Caps
713,"Long-Sleeve Logo Jersey, S",LJ-0192-S,False,Multi,4,3,38.4923,49.99,S,null,null,null,null,Jerseys
714,"Long-Sleeve Logo Jersey, M",LJ-0192-M,False,Multi,4,3,38.4923,49.99,M,null,null,null,null,Jerseys


In [0]:
display(raw_sales_order_detail_df.limit(10))

SalesOrderID,SalesOrderDetailID,OrderQty,ProductID,UnitPrice,UnitPriceDiscount
43659,1,1,776,2024.9940,.0000
43659,2,3,777,2024.9940,.0000
43659,3,1,778,2024.9940,.0000
43659,4,1,771,2039.9940,.0000
43659,5,1,772,2039.9940,.0000
43659,6,2,773,2039.9940,.0000
43659,7,1,774,2039.9940,.0000
43659,8,3,714,28.8404,.0000
43659,9,1,716,28.8404,.0000
43659,10,6,709,5.7000,.0000


In [0]:
display(raw_sales_order_header_df.limit(10))

SalesOrderID,OrderDate,ShipDate,OnlineOrderFlag,AccountNumber,CustomerID,SalesPersonID,Freight
43828,2021-06,2021-07-05,True,10-4030-027605,27605,null,89.4568
43829,2021-06,2021-07-05,True,10-4030-027611,27611,null,89.4568
43830,2021-06,2021-07-05,True,10-4030-016347,16347,null,89.4568
43831,2021-06,2021-07-05,True,10-4030-011028,11028,null,84.3748
43832,2021-06,2021-07-05,True,10-4030-013584,13584,null,89.4568
43659,2021-05-31,2021-06-07,False,10-4020-000676,29825,279,616.0984
43660,2021-05-31,2021-06-07,False,10-4020-000117,29672,279,38.8276
43661,2021-05-31,2021-06-07,False,10-4020-000442,29734,282,985.553
43662,2021-05-31,2021-06-07,False,10-4020-000227,29994,282,867.2389
43663,2021-05-31,2021-06-07,False,10-4020-000510,29565,276,12.5838


In [0]:
(
    raw_products_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("raw_products")
)

(
    raw_sales_order_detail_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("raw_sales_order_detail")
)

(
    raw_sales_order_header_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("raw_sales_order_header")
)

In [0]:
order_date_raw = col("OrderDate").cast("string")
ship_date = to_date(col("ShipDate").cast("string"))

is_order_year_month = order_date_raw.rlike(r"^\d{4}-\d{2}$")

order_month_date = to_date(concat(order_date_raw, lit("-01")))
ship_month_date = trunc(ship_date, "MM")

treated_sales_order_header_df = (
    raw_sales_order_header_df
    .withColumn(
        "OrderDate",
        when(
            is_order_year_month,
            when(
                ship_month_date == order_month_date,
                order_month_date
            )
            .when(
                ship_month_date > order_month_date,
                last_day(order_month_date)
            )
            .otherwise(order_month_date)
        )
        .otherwise(to_date(order_date_raw))
    )
)

In [0]:
display(treated_sales_order_header_df.limit(10))

SalesOrderID,OrderDate,ShipDate,OnlineOrderFlag,AccountNumber,CustomerID,SalesPersonID,Freight
43828,2021-06-30,2021-07-05,True,10-4030-027605,27605,null,89.4568
43829,2021-06-30,2021-07-05,True,10-4030-027611,27611,null,89.4568
43830,2021-06-30,2021-07-05,True,10-4030-016347,16347,null,89.4568
43831,2021-06-30,2021-07-05,True,10-4030-011028,11028,null,84.3748
43832,2021-06-30,2021-07-05,True,10-4030-013584,13584,null,89.4568
43659,2021-05-31,2021-06-07,False,10-4020-000676,29825,279,616.0984
43660,2021-05-31,2021-06-07,False,10-4020-000117,29672,279,38.8276
43661,2021-05-31,2021-06-07,False,10-4020-000442,29734,282,985.553
43662,2021-05-31,2021-06-07,False,10-4020-000227,29994,282,867.2389
43663,2021-05-31,2021-06-07,False,10-4020-000510,29565,276,12.5838


In [0]:
store_products_df = raw_products_df.select(
    col("ProductID").cast("int").alias("ProductID"),
    col("ProductDesc").cast("string").alias("ProductDesc"),
    col("ProductNumber").cast("string").alias("ProductNumber"),
    col("MakeFlag").cast("boolean").alias("MakeFlag"),
    col("Color").cast("string").alias("Color"),
    col("SafetyStockLevel").cast("int").alias("SafetyStockLevel"),
    col("ReorderPoint").cast("int").alias("ReorderPoint"),
    col("StandardCost").cast("double").alias("StandardCost"),
    col("ListPrice").cast("double").alias("ListPrice"),
    col("Size").cast("string").alias("Size"),
    col("SizeUnitMeasureCode").cast("string").alias("SizeUnitMeasureCode"),
    col("Weight").cast("double").alias("Weight"),
    col("WeightUnitMeasureCode").cast("string").alias("WeightUnitMeasureCode"),
    col("ProductCategoryName").cast("string").alias("ProductCategoryName"),
    col("ProductSubCategoryName").cast("string").alias("ProductSubCategoryName")
)

store_sales_order_detail_df = raw_sales_order_detail_df.select(
    col("SalesOrderID").cast("int").alias("SalesOrderID"),
    col("SalesOrderDetailID").cast("int").alias("SalesOrderDetailID"),
    col("OrderQty").cast("int").alias("OrderQty"),
    col("ProductID").cast("int").alias("ProductID"),
    col("UnitPrice").cast("double").alias("UnitPrice"),
    col("UnitPriceDiscount").cast("double").alias("UnitPriceDiscount")
)

store_sales_order_header_df = treated_sales_order_header_df.select(
    col("SalesOrderID").cast("int").alias("SalesOrderID"),
    col("OrderDate").cast("date").alias("OrderDate"),
    col("ShipDate").cast("date").alias("ShipDate"),
    col("OnlineOrderFlag").cast("boolean").alias("OnlineOrderFlag"),
    col("AccountNumber").cast("string").alias("AccountNumber"),
    col("CustomerID").cast("int").alias("CustomerID"),
    col("SalesPersonID").cast("int").alias("SalesPersonID"),
    col("Freight").cast("double").alias("Freight")
)

In [0]:
display(store_products_df.limit(10))

ProductID,ProductDesc,ProductNumber,MakeFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,Weight,WeightUnitMeasureCode,ProductCategoryName,ProductSubCategoryName
680,"HL Road Frame - Black, 58",FR-R92B-58,true,Black,500,375,1059.31,1431.5,58,CM,2.24,LB,null,Road Frames
706,"HL Road Frame - Red, 58",FR-R92R-58,true,Red,500,375,1059.31,1431.5,58,CM,2.24,LB,null,Road Frames
707,"Sport-100 Helmet, Red",HL-U509-R,false,Red,4,3,13.0863,34.99,null,null,null,null,null,Helmets
708,"Sport-100 Helmet, Black",HL-U509,false,Black,4,3,13.0863,34.99,null,null,null,null,null,Helmets
709,"Mountain Bike Socks, M",SO-B909-M,false,White,4,3,3.3963,9.5,M,null,null,null,null,Socks
710,"Mountain Bike Socks, L",SO-B909-L,false,White,4,3,3.3963,9.5,L,null,null,null,null,Socks
711,"Sport-100 Helmet, Blue",HL-U509-B,false,Blue,4,3,13.0863,34.99,null,null,null,null,null,Helmets
712,AWC Logo Cap,CA-1098,false,Multi,4,3,6.9223,8.99,null,null,null,null,null,Caps
713,"Long-Sleeve Logo Jersey, S",LJ-0192-S,false,Multi,4,3,38.4923,49.99,S,null,null,null,null,Jerseys
714,"Long-Sleeve Logo Jersey, M",LJ-0192-M,false,Multi,4,3,38.4923,49.99,M,null,null,null,null,Jerseys


In [0]:
display(store_sales_order_detail_df.limit(10))

SalesOrderID,SalesOrderDetailID,OrderQty,ProductID,UnitPrice,UnitPriceDiscount
43659,1,1,776,2024.994,0.0
43659,2,3,777,2024.994,0.0
43659,3,1,778,2024.994,0.0
43659,4,1,771,2039.994,0.0
43659,5,1,772,2039.994,0.0
43659,6,2,773,2039.994,0.0
43659,7,1,774,2039.994,0.0
43659,8,3,714,28.8404,0.0
43659,9,1,716,28.8404,0.0
43659,10,6,709,5.7,0.0


In [0]:
display(store_sales_order_header_df.limit(10))

SalesOrderID,OrderDate,ShipDate,OnlineOrderFlag,AccountNumber,CustomerID,SalesPersonID,Freight
43828,2021-06-30,2021-07-05,true,10-4030-027605,27605,null,89.4568
43829,2021-06-30,2021-07-05,true,10-4030-027611,27611,null,89.4568
43830,2021-06-30,2021-07-05,true,10-4030-016347,16347,null,89.4568
43831,2021-06-30,2021-07-05,true,10-4030-011028,11028,null,84.3748
43832,2021-06-30,2021-07-05,true,10-4030-013584,13584,null,89.4568
43659,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984
43660,2021-05-31,2021-06-07,false,10-4020-000117,29672,279,38.8276
43661,2021-05-31,2021-06-07,false,10-4020-000442,29734,282,985.553
43662,2021-05-31,2021-06-07,false,10-4020-000227,29994,282,867.2389
43663,2021-05-31,2021-06-07,false,10-4020-000510,29565,276,12.5838


In [0]:
(
    store_products_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("store_products")
)

(
    store_sales_order_detail_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("store_sales_order_detail")
)

(
    store_sales_order_header_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("store_sales_order_header")
)


### Data Quality Checks

This section performs baseline data quality validations to ensure the curated tables are reliable for downstream transformations and analysis.

The checks focus on:

* Primary key uniqueness
* Referential integrity between related tables
* Detection of orphan records


In [0]:
%sql
SELECT 
  COUNT(1) AS total_rows,
  COUNT(DISTINCT ProductID) AS distinct_product_ids
FROM store_products;

total_rows,distinct_product_ids
303,295


In [0]:
%sql
SELECT 
  COUNT(1) AS total_rows,
  COUNT(DISTINCT SalesOrderID) AS distinct_sales_order_ids
FROM store_sales_order_header;

total_rows,distinct_sales_order_ids
31465,31465


In [0]:
%sql
SELECT 
  COUNT(1) AS total_rows,
  COUNT(DISTINCT SalesOrderDetailID) AS distinct_sales_order_detail_ids
FROM store_sales_order_detail;

total_rows,distinct_sales_order_detail_ids
121317,121317


In [0]:
%sql
SELECT COUNT(1) AS orphan_sales_order_detail_rows
FROM store_sales_order_detail d
LEFT JOIN store_sales_order_header h
  ON d.SalesOrderID = h.SalesOrderID
WHERE h.SalesOrderID IS NULL;

orphan_sales_order_detail_rows
0


In [0]:
%sql
SELECT COUNT(1) AS orphan_product_rows
FROM store_sales_order_detail d
LEFT JOIN store_products p
  ON d.ProductID = p.ProductID
WHERE p.ProductID IS NULL;

orphan_product_rows
0



### Product Master Deduplication

During the data quality checks, duplicated `ProductID` records were identified in the product master table, although `ProductID` is expected to uniquely identify each product.

As the definitive business rule for handling these duplicates is still unclear, this should be validated with the business team. To avoid blocking downstream work, a temporary deduplication rule was applied using `ROW_NUMBER()`.

The selected rule prioritizes records with populated `ProductCategoryName` and `ProductSubCategoryName`, preserving the most complete product classification available for downstream transformations and analysis.


In [0]:
product_dedup_window = (
    Window
    .partitionBy("ProductID")
    .orderBy(
        when(col("ProductCategoryName").isNotNull(), 1).otherwise(0).desc(),
        when(col("ProductSubCategoryName").isNotNull(), 1).otherwise(0).desc(),
        col("ProductID")
    )
)

store_products_df = (
    raw_products_df
    .withColumn("row_num", row_number().over(product_dedup_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

(
    store_products_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("store_products")
)

In [0]:
%sql
SELECT 
  COUNT(1) AS total_rows,
  COUNT(DISTINCT ProductID) AS distinct_product_ids
FROM store_products;

total_rows,distinct_product_ids
295,295


### Primary and Foreign Keys

#### store_products
- Primary Key: ProductID

#### store_sales_order_header
- Primary Key: SalesOrderID

#### store_sales_order_detail
- Primary Key: SalesOrderDetailID
- Foreign Key: SalesOrderID references store_sales_order_header.SalesOrderID
- Foreign Key: ProductID references store_products.ProductID

In [0]:
publish_product_df = (
    store_products_df
    .withColumn(
        "Color",
        when(col("Color").isNull(), lit("N/A"))
        .otherwise(col("Color"))
    )
    .withColumn(
        "ProductCategoryName",
        when(
            col("ProductCategoryName").isNull() &
            col("ProductSubCategoryName").isin("Gloves", "Shorts", "Socks", "Tights", "Vests"),
            lit("Clothing")
        )
        .when(
            col("ProductCategoryName").isNull() &
            col("ProductSubCategoryName").isin("Locks", "Lights", "Headsets", "Helmets", "Pedals", "Pumps"),
            lit("Accessories")
        )
        .when(
            col("ProductCategoryName").isNull() &
            (
                col("ProductSubCategoryName").contains("Frames") |
                col("ProductSubCategoryName").isin("Wheels", "Saddles")
            ),
            lit("Components")
        )
        .otherwise(col("ProductCategoryName"))
    )
)
(
    publish_product_df.createOrReplaceTempView("publish_product")
)

In [0]:
display(publish_product_df.limit(10))

ProductID,ProductDesc,ProductNumber,MakeFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,Weight,WeightUnitMeasureCode,ProductCategoryName,ProductSubCategoryName
680,"HL Road Frame - Black, 58",FR-R92B-58,True,Black,500,375,1059.31,1431.5,58,CM,2.24,LB,Components,Road Frames
706,"HL Road Frame - Red, 58",FR-R92R-58,True,Red,500,375,1059.31,1431.5,58,CM,2.24,LB,Components,Road Frames
707,"Sport-100 Helmet, Red",HL-U509-R,False,Red,4,3,13.0863,34.99,null,null,null,null,Accessories,Helmets
708,"Sport-100 Helmet, Black",HL-U509,False,Black,4,3,13.0863,34.99,null,null,null,null,Accessories,Helmets
709,"Mountain Bike Socks, M",SO-B909-M,False,White,4,3,3.3963,9.5,M,null,null,null,Clothing,Socks
710,"Mountain Bike Socks, L",SO-B909-L,False,White,4,3,3.3963,9.5,L,null,null,null,Clothing,Socks
711,"Sport-100 Helmet, Blue",HL-U509-B,False,Blue,4,3,13.0863,34.99,null,null,null,null,Accessories,Helmets
712,AWC Logo Cap,CA-1098,False,Multi,4,3,6.9223,8.99,null,null,null,null,null,Caps
713,"Long-Sleeve Logo Jersey, S",LJ-0192-S,False,Multi,4,3,38.4923,49.99,S,null,null,null,Clothing,Shirt
714,"Long-Sleeve Logo Jersey, M",LJ-0192-M,False,Multi,4,3,38.4923,49.99,M,null,null,null,Clothing,Shirt


In [0]:
%sql
CREATE OR REPLACE VIEW publish_orders AS
SELECT
    d.SalesOrderID,
    d.SalesOrderDetailID,
    d.OrderQty,
    d.ProductID,
    d.UnitPrice,
    d.UnitPriceDiscount,

    h.OrderDate,
    h.ShipDate,
    h.OnlineOrderFlag,
    h.AccountNumber,
    h.CustomerID,
    h.SalesPersonID,
    h.Freight AS TotalOrderFreight,

    CASE
        WHEN h.OrderDate IS NULL OR h.ShipDate IS NULL THEN NULL
        WHEN h.ShipDate <= TO_DATE(h.OrderDate) THEN 0
        ELSE SIZE(
            FILTER(
                SEQUENCE(TO_DATE(h.OrderDate), DATE_SUB(h.ShipDate, 1)),
                business_day -> DAYOFWEEK(business_day) BETWEEN 2 AND 6
            )
        )
    END AS LeadTimeInBusinessDays,

    d.OrderQty * (d.UnitPrice - d.UnitPriceDiscount) AS TotalLineExtendedPrice

FROM store_sales_order_detail d
INNER JOIN store_sales_order_header h
    ON d.SalesOrderID = h.SalesOrderID;

In [0]:
%sql

SELECT * FROM publish_orders
limit 10

SalesOrderID,SalesOrderDetailID,OrderQty,ProductID,UnitPrice,UnitPriceDiscount,OrderDate,ShipDate,OnlineOrderFlag,AccountNumber,CustomerID,SalesPersonID,TotalOrderFreight,LeadTimeInBusinessDays,TotalLineExtendedPrice
43659,1,1,776,2024.994,0.0,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,2024.994
43659,2,3,777,2024.994,0.0,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,6074.982
43659,3,1,778,2024.994,0.0,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,2024.994
43659,4,1,771,2039.994,0.0,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,2039.994
43659,5,1,772,2039.994,0.0,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,2039.994
43659,6,2,773,2039.994,0.0,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,4079.988
43659,7,1,774,2039.994,0.0,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,2039.994
43659,8,3,714,28.8404,0.0,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,86.5212
43659,9,1,716,28.8404,0.0,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,28.8404
43659,10,6,709,5.7,0.0,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,34.2


####Analysis question 1: Which color generated the highest revenue each year?

In [0]:
%sql
WITH yearly_color_revenue AS (
    SELECT
        YEAR(o.OrderDate) AS OrderYear,
        p.Color,
        SUM(o.TotalLineExtendedPrice) AS TotalRevenue
    FROM publish_orders o
    INNER JOIN publish_product p
        ON o.ProductID = p.ProductID
    GROUP BY
        YEAR(o.OrderDate),
        p.Color
)

SELECT
    OrderYear,
    Color,
    ROUND(TotalRevenue, 2) AS TotalRevenue
FROM yearly_color_revenue
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY OrderYear
    ORDER BY TotalRevenue DESC
) = 1
ORDER BY OrderYear;

OrderYear,Color,TotalRevenue
2021,Red,6019614.02
2022,Black,1.400524298E7
2023,Black,1.504769437E7
2024,Yellow,6368158.48


####Analysis question 2: Average LeadTimeInBusinessDays by ProductCategoryName

In [0]:
%sql
SELECT
    p.ProductCategoryName,
    ROUND(AVG(o.LeadTimeInBusinessDays), 0) AS AverageLeadTimeInBusinessDays
FROM publish_orders o
INNER JOIN publish_product p
    ON o.ProductID = p.ProductID
    WHERE ProductCategoryName IS NOT NULL
GROUP BY p.ProductCategoryName
ORDER BY AverageLeadTimeInBusinessDays DESC;

ProductCategoryName,AverageLeadTimeInBusinessDays
Bikes,5.0
Clothing,5.0
Accessories,5.0
Components,5.0
